## CS310 Natural Language Processing
## Assignment 2. Word2vec Implementation 

**Total points**: 30

Train a word2vec model using the **skip-gram** architecture and **negative sampling**.

You should roughtly follow the structure of the notebook. Add additional cells if you feel needed. 

You can (and you should) re-use the code from *Lab 3: Data preparation for implementing word2vec*. 

Make sure your code is readable and well-structured.

## 0. Import Necessary Libraries

In [ ]:
from typing import List
from utils import CorpusReader
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

## 1. Data Processing

The corpus data is in `shakespeare.txt`. Use the `CorpusReader` class in `utils.py` to help you.

In [ ]:
### YOUR CODE HERE ###
#初始化CorpusReader
corpus = CorpusReader('shakespeare.txt', min_count=3, lang='en')
print(f"词表大小: {corpus.vocab_size}")

#要一个包含所有分词后的列表方便generate_data函数调用
all_words = []
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    for line in f:
        words = line.strip().split()
        if words:
            all_words.extend(words)
### END YOUR CODE ###

In [ ]:
# Re-use the code from lab with necessary modifications

def generate_data(words: List[str], window_size: int, k: int, corpus: CorpusReader):
    """ Generate the training data for word2vec skip-gram model
    Args:
        text: the input text
        window_size: the size of the context window
        k: the number of negative samples
        corpus: the corpus object, providing utilities such as word2id, getNegatives, etc.
    """
    ### YOUR CODE HERE ###
    data = []
    #将单词序列转换为 ID 序列，如果在词表中（min_count过滤后的）则保留
    word_ids = [corpus.word2id[w] for w in words if w in corpus.word2id]
    
    for i in range(len(word_ids)):
        center_id = word_ids[i]
        
        #确定窗口左右边界 不要越界
        start = max(0, i - window_size)
        end = min(len(word_ids), i + window_size + 1)
        
        for j in range(start, end):
            if i == j: continue #跳过自己
            
            outside_id = word_ids[j]
            #获取k个负样本ID
            negative_ids = corpus.getNegatives(outside_id, k)
            
            data.append((center_id, outside_id, negative_ids))
            
    return data
    ### END YOUR CODE ###

def batchify(data: List, batch_size: int):
    """ Group a stream into batches and yield them as torch tensors.
    Args:
        data: a list of tuples
        batch_size: the batch size 
    Yields:
        a tuple of three torch tensors: center, outside, negative
    """
    assert batch_size < len(data) # data should be long enough
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        if i > len(data) - batch_size: # if the last batch is smaller than batch_size, pad it with the first few data
            batch = batch + data[:i + batch_size - len(data)]
        
        ### YOUR CODE HERE ###
        # 将目前的tuple list解包转为longtensor
        #centers: (batch_size, )
        centers = torch.LongTensor([item[0] for item in batch])
        #outsides: (batch_size, )
        outsides = torch.LongTensor([item[1] for item in batch])
        #negatives: (batch_size, k)
        negatives = torch.LongTensor([item[2] for item in batch])
        
        yield centers, outsides, negatives
        ### END YOUR CODE ###

## 2. Define the Model

In [ ]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super(SkipGram, self).__init__()
        self.vocab_size = vocab_size
        self.emb_size = emb_size
        self.emb_v = nn.Embedding(vocab_size, emb_size, sparse=True)
        self.emb_u = nn.Embedding(vocab_size, emb_size, sparse=True)

        initrange = 1.0 / self.emb_size # some experience passed down from generation to generation
        nn.init.uniform_(self.emb_v.weight.data, -initrange, initrange) # same outcome as self.emb_v.weight.data.uniform_(-initrange, initrange)
        nn.init.constant_(self.emb_u.weight.data, 0) # same outcome as self.emb_u.weight.data.zero_()

    def forward(self, center, outside, negative):
        """
        Args:
            center: the center word indices (B, )
            outside: the outside word indices (B, )
            negative: the negative word indices (B, k)
        """
        v_c = self.emb_v(center)
        u_o = self.emb_u(outside)
        u_n = self.emb_u(negative)
        
        ### YOUR CODE HERE ###
        #计算正样本评分：
        loss = None
        pos_score = torch.sum(v_c * u_o, dim=1)  # (B, )
        #计算负样本评分
        neg_scores = torch.bmm(u_n, v_c.unsqueeze(2)).squeeze(2) # (B, k)
        #计算对数sigmoid损失
        #正样本损失: log(sigma(v_c · u_o))
        pos_loss = F.logsigmoid(torch.clamp(pos_score, min=-10, max=10))
        #负样本损失: sum(log(sigma(-v_c · u_n)))
        eg_loss = torch.sum(F.logsigmoid(torch.clamp(-neg_scores, min=-10, max=10)), dim=1) # (B, )
        #合并损失取平均值
        loss = -torch.mean(pos_loss + neg_loss)
        # Hint: torch.clamp the input to F.logsigmoid to avoid numerical underflow/overflow
        ### END YOUR CODE ###

        return loss
    
    def save_embedding(self, id2word, file_name):
        embedding = self.emb_v.weight.cpu().data.numpy()
        with open(file_name, 'w') as f:
            f.write('%d %d\n' % (len(id2word), self.emb_size))
            for wid, w in id2word.items():
                e = ' '.join(map(lambda x: str(x), embedding[wid]))
                f.write('%s %s\n' % (w, e))

## 3. Train and Evaluate

In [ ]:
def train(model, dataloader, optimizer, epochs):
    # Write your own code for this train function
    # You don't need exactly the same arguments

    ### YOUR CODE HERE ###
    #保存 loss 历史以便绘图
    loss_history = []
    
    #获取运行设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train() #设置为训练模式
    for epoch in range(epochs):
        total_loss = 0
        for i, (center, outside, negative) in enumerate(dataloader):
            # 将张量搬移到 GPU/CPU
            center, outside, negative = center.to(device), outside.to(device), negative.to(device)
            optimizer.zero_grad()    #梯度归零
            loss = model(center, outside, negative) #前向传播
            loss.backward()          # 反向传播
            
            #梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()#更新参数
            
            total_loss += loss.item()
            
            if (i + 1) % 1000 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}], Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / (i + 1)
        loss_history.append(avg_loss)
        print(f"Epoch [{epoch+1}/{epochs}] 完成，平均 Loss: {avg_loss:.4f}")
    
    return loss_history

    ### END YOUR CODE ###


# Suggested hyperparameters
initial_lr = 0.025
batch_size = 16
emb_size = 100
window_size = 3
k = 5 # the number of negative samples
min_count = 3 # because our data is small. If min_count > 1, you should filter out those unknown words 
optimizer = torch.optim.Adam() # or torch.optim.SparseAdam()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR() # or torch.optim.lr_scheduler.StepLR()

# Initialize the corpus and model
corpus = CorpusReader('YOUR_INPUT.txt', min_count)
vocab_size = corpus.vocab_size
model = SkipGram(vocab_size, emb_size)


### Hints: ###
# - If you have cuda-supported GPUs, you can run the training faster by
#   `device = torch.device("cuda" if self.use_cuda else "cpu")`
#   `model.cuda()`
#   You also need to move all tensor data to the same device
# - If you find Inf or NaN in the loss, you can try to clip the gradient usning `torch.nn.utils.clip_grad_norm_`
# - Remember to save the embeddings when training is done

## 4. Save the Embeddings


Save the embeddings into a `gensim` compatible format.

In [ ]:
output_file = 'embeddings.txt'

weights = model.emb_v.detech().cpu().data.numpy()
with open(output_file, "w") as f:
    f.write(f"{vocab_size} {emb_size}\n")  # First line: vocab size and vector dimension
    for idx, vector in enumerate(weights):
        vector_str = " ".join(map(str, vector))
        f.write(f"{model.id2word[idx]} {vector_str}\n") 

## 5. Plot and Compare Embeddings

Use `sklearn.decomposition.TruncatedSVD` to reduce the dimensionality of the obtained embeddings to 2 and plot the selected words in 2D space.

*Hint*:
- Obtain the embeddings into a numpy array by `model.emb_v.cpu().data.numpy()`
- The word2id dictionary is in `model.word2id`
- If you are trying to load from a saved embedding file, you can use the APIs from `gensim`.
  - For exmaple, `model = gensim.models.KeyedVectors.load_word2vec_format('path/to/file')`
  - Check out the documentation for more details: https://radimrehurek.com/gensim/models/keyedvectors.html

In [ ]:
# Load embeddings
### YOUR CODE HERE ###
#从模型的 emb_v 层提取参数并转为 numpy 数组
all_embeddings = model.emb_v.weight.detach().cpu().numpy()
#获取 word 到 id 的映射
word2id = corpus.word2id
### END YOUR CODE ###

In [ ]:
# Truncated SVD 降维处理
### YOUR CODE HERE ###
#获取 word 到 id 的映射
#初始化 SVD 模型 n_components制定维度
svd = TruncatedSVD(n_components=2)
#对全量词向量进行拟合与转换
#(vocab_size, 2)
embeddings_2d = svd.fit_transform(all_embeddings)
print(f"降维后的数据形状: {embeddings_2d.shape}")
### END YOUR CODE ###

In [ ]:
# Plot the following words or other words you are interested in
# You better pick those words that look different in the 2D space compared with the LSA vectors
words = ['king', 'queen', 'man', 'woman', 'romeo', 'juliet', 'life', 'death']

### YOUR CODE HERE ###
plt.figure(figsize=(10, 8))
for word in words:
    #检查单词是否在词表中
    if word in word2id:
        idx = word2id[word]
        #获取该词在 2D 空间中的坐标
        x, y = embeddings_2d[idx]
        
        #绘制点
        plt.scatter(x, y, color='red')
        #标注单词文本
        plt.annotate(word, (x, y), xytext=(5, 2), textcoords='offset points', fontsize=12)
    else:
        print(f"Warning: '{word}' is not in vocabulary.")
plt.title('Word2Vec Visualisation using Truncated SVD')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True)
plt.show()
### END YOUR CODE ###